In [1]:
import os

In [2]:
%pwd

'f:\\nlp\\Text-Summarizer-Project\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'f:\\nlp\\Text-Summarizer-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    per_device_eval_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_steps: int
    eval_strategy: str

In [6]:
from textSummarizer.constants import * 
from textSummarizer.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):
        self.config_filepath = Path(config_filepath)
        self.params_filepath = Path(params_filepath)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root], verbose=True)
    
    def get_model_trainer_config(self)-> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments
        create_directories([config.root_dir], verbose=True)
        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            per_device_eval_batch_size=params.per_device_eval_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            eval_steps=params.eval_steps,
            eval_strategy=params.eval_strategy,
        )
        return model_trainer_config
        


In [8]:
from transformers import AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from transformers import AutoTokenizer
from datasets import load_from_disk,DatasetDict
import os

class ModelTrainer:
    def __init__(self, config):
        self.config = config

    def train(self):
        # ✅ Load full dataset
        dataset = load_from_disk(self.config.data_path)

        # ✅ Load model and tokenizer
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt)
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)

        # ✅ Prepare data collator
        data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
        training_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.eval_strategy,
            eval_steps=self.config.eval_steps,
            save_total_limit=1,
            report_to="none"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            tokenizer=tokenizer,
            data_collator=data_collator,
            train_dataset=dataset["train"],
            eval_dataset=dataset["validation"]
        )

        trainer.train()
        model.save_pretrained(os.path.join(self.config.root_dir, "t5-small_model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))


In [9]:
try:
    configuration_manager = ConfigurationManager()
    model_trainer_config = configuration_manager.get_model_trainer_config()
    model_trainer = ModelTrainer(model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2025-07-30 11:01:24,347:INFO:yaml file: config\config.yaml loaded sucessfully]
[2025-07-30 11:01:24,357:INFO:yaml file: params.yaml loaded sucessfully]
[2025-07-30 11:01:24,358:INFO:created directory at: artifacts]
[2025-07-30 11:01:24,360:INFO:created directory at: artifacts/model_trainer]


  0%|          | 0/44196 [00:00<?, ?it/s]

{'loss': 12.6775, 'grad_norm': 98.40135955810547, 'learning_rate': 2.5e-05, 'epoch': 0.01}
{'loss': 2.1672, 'grad_norm': 8.257797241210938, 'learning_rate': 5e-05, 'epoch': 0.01}
{'loss': 0.7602, 'grad_norm': 5.220596790313721, 'learning_rate': 4.988635330484589e-05, 'epoch': 0.02}
{'loss': 0.6567, 'grad_norm': 2.625178575515747, 'learning_rate': 4.9772706609691796e-05, 'epoch': 0.03}
{'loss': 0.5677, 'grad_norm': 1.7635741233825684, 'learning_rate': 4.965905991453769e-05, 'epoch': 0.03}


  0%|          | 0/818 [00:00<?, ?it/s]

{'eval_loss': 0.4757465720176697, 'eval_runtime': 60.2991, 'eval_samples_per_second': 13.566, 'eval_steps_per_second': 13.566, 'epoch': 0.03}
{'loss': 0.568, 'grad_norm': 1.8074946403503418, 'learning_rate': 4.9545413219383584e-05, 'epoch': 0.04}
{'loss': 0.5283, 'grad_norm': 2.245205879211426, 'learning_rate': 4.943176652422948e-05, 'epoch': 0.05}
{'loss': 0.4712, 'grad_norm': 1.589501142501831, 'learning_rate': 4.931811982907537e-05, 'epoch': 0.05}
{'loss': 0.5105, 'grad_norm': 1.6250102519989014, 'learning_rate': 4.920447313392127e-05, 'epoch': 0.06}
{'loss': 0.4909, 'grad_norm': 4.177312850952148, 'learning_rate': 4.9090826438767165e-05, 'epoch': 0.07}


...

{'eval_loss': 0.37577420473098755, 'eval_runtime': 59.6546, 'eval_samples_per_second': 13.712, 'eval_steps_per_second': 13.712, 'epoch': 2.92}
{'loss': 0.3917, 'grad_norm': 1.9395256042480469, 'learning_rate': 1.24556777888899e-06, 'epoch': 2.93}
{'loss': 0.402, 'grad_norm': 0.9839732050895691, 'learning_rate': 1.131921083734885e-06, 'epoch': 2.93}
{'loss': 0.3887, 'grad_norm': 1.6079294681549072, 'learning_rate': 1.0182743885807801e-06, 'epoch': 2.94}
{'loss': 0.3948, 'grad_norm': 2.2248542308807373, 'learning_rate': 9.046276934266752e-07, 'epoch': 2.95}
{'loss': 0.3761, 'grad_norm': 1.061145305633545, 'learning_rate': 7.909809982725704e-07, 'epoch': 2.95}


  0%|          | 0/818 [00:00<?, ?it/s]

{'eval_loss': 0.3759819269180298, 'eval_runtime': 60.8987, 'eval_samples_per_second': 13.432, 'eval_steps_per_second': 13.432, 'epoch': 2.95}
{'loss': 0.3853, 'grad_norm': 1.287170171737671, 'learning_rate': 6.773343031184654e-07, 'epoch': 2.96}
{'loss': 0.4051, 'grad_norm': 2.8510446548461914, 'learning_rate': 5.636876079643604e-07, 'epoch': 2.97}
{'loss': 0.368, 'grad_norm': 2.156383752822876, 'learning_rate': 4.500409128102555e-07, 'epoch': 2.97}
{'loss': 0.4033, 'grad_norm': 1.2983152866363525, 'learning_rate': 3.3639421765615053e-07, 'epoch': 2.98}
{'loss': 0.4144, 'grad_norm': 2.061746597290039, 'learning_rate': 2.2274752250204563e-07, 'epoch': 2.99}


  0%|          | 0/818 [00:00<?, ?it/s]

{'eval_loss': 0.3758583068847656, 'eval_runtime': 68.4928, 'eval_samples_per_second': 11.943, 'eval_steps_per_second': 11.943, 'epoch': 2.99}
{'loss': 0.3596, 'grad_norm': 2.579775333404541, 'learning_rate': 1.0910082734794073e-07, 'epoch': 2.99}
{'train_runtime': 20022.6403, 'train_samples_per_second': 2.207, 'train_steps_per_second': 2.207, 'train_loss': 0.4518716890506156, 'epoch': 3.0}
